In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# Data Generation & Loading into Data Frame

In [3]:
np.random.seed(0)
n = 120

data = {
    'age_years':   np.random.randint(1, 15, n),
    'km_driven':   np.random.randint(5000, 150000, n),
    'engine_cc':   np.random.randint(800, 3500, n),
    'fuel_type':   np.random.randint(0, 2, n),
    'noise_1':     np.random.randn(n),
    'noise_2':     np.random.randn(n),
}
df = pd.DataFrame(data)
df['price_lakhs'] = (
    -0.5  * df['age_years']
    - 0.00003 * df['km_driven']
    + 0.003 * df['engine_cc']
    + 2.0  * df['fuel_type']
    + np.random.randn(n) * 2
)

X = df.drop('price_lakhs', axis=1)
y = df['price_lakhs']

# Task 1 — Baseline and Diagnosis

Split 80/20 with random_state=42, scale with StandardScaler on training only, and train LinearRegression. Print train R² and test R². In a comment, state whether the gap indicates overfitting, underfitting, or a good fit.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
print("Task 1: Baseline")
print(f"Train R²: {lr.score(X_train_scaled, y_train):.4f}")
print(f"Test R²: {lr.score(X_test_scaled, y_test):.4f}")

Task 1: Baseline
Train R²: 0.6856
Test R²: 0.6365


In [ ]:
# Small gap between Train and Test R² indicates a good fit.

# Task 2 — Ridge Tuning

Loop through alphas [0.01, 1, 10, 100, 500], train a Ridge model for each, and print train R² and test R². Print the best alpha. In a comment, explain what happens to bias and variance as alpha increases.

In [8]:
alphas = [0.01, 1, 10, 100, 500]
print("\nTask 2: Ridge Tuning")
for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train_scaled, y_train)
    print(f"Alpha {a:>4}: Train R² = {ridge.score(X_train_scaled, y_train):.4f}, "
          f"Test R² = {ridge.score(X_test_scaled, y_test):.4f}")



Task 2: Ridge Tuning
Alpha 0.01: Train R² = 0.6856, Test R² = 0.6365
Alpha    1: Train R² = 0.6855, Test R² = 0.6405
Alpha   10: Train R² = 0.6788, Test R² = 0.6637
Alpha  100: Train R² = 0.5052, Test R² = 0.5549
Alpha  500: Train R² = 0.2064, Test R² = 0.2335


In [ ]:
# Best Alpha: 10
# Increasing alpha increases bias (simplifies model) and decreases variance (less overfitting).

# Task 3 — Lasso Feature Selection

Train a Lasso model with alpha=1.0 and max_iter=10000. Print each feature's coefficient. In a comment, identify which features were zeroed out and what that means.

In [9]:
lasso = Lasso(alpha=1.0, max_iter=10000)
lasso.fit(X_train_scaled, y_train)
print("\nTask 3: Lasso Coefficients")
for feature, coef in zip(X.columns, lasso.coef_):
    print(f"{feature:10}: {coef:.4f}")


Task 3: Lasso Coefficients
age_years : -0.6307
km_driven : -0.2007
engine_cc : 1.0973
fuel_type : 0.0000
noise_1   : 0.0000
noise_2   : 0.0000


In [ ]:
# fuel_type, noise_1, and noise_2 were zeroed out.
# It means Lasso dropped them as non-significant predictors

# Task 4 — Validation Stability

Re-run your best Ridge model using three different random seeds [42, 7, 123] for the train-test split. Print test R² for each seed. In a comment, state whether the small or large variation confirms model stability.

In [10]:
seeds = [42, 7, 123]
print("\nTask 4: Stability (Ridge alpha=10)")
for s in seeds:
    X_s_tr, X_s_ts, y_s_tr, y_s_ts = train_test_split(X, y, test_size=0.20, random_state=s)
    sc = StandardScaler()
    X_s_tr_sc = sc.fit_transform(X_s_tr)
    X_s_ts_sc = sc.transform(X_s_ts)

    ridge_stab = Ridge(alpha=10)
    ridge_stab.fit(X_s_tr_sc, y_s_tr)
    print(f"Seed {s:>3}: Test R² = {ridge_stab.score(X_s_ts_sc, y_s_ts):.4f}")


Task 4: Stability (Ridge alpha=10)
Seed  42: Test R² = 0.6637
Seed   7: Test R² = 0.6649
Seed 123: Test R² = 0.6026


In [ ]:
# Small variations across different seeds confirm the model is stable.